# SceneProof — GPU(TensorRT) 측정 (Google Colab)

**무엇을 하나요?**
1. 사람 찾기 모델(YOLOX nano·tiny·s)을 **같은 영상 프레임, 같은 전처리·후처리**로 다섯 가지 방식으로 돌려 속도와 결과 차이를 잽니다:
   CPU(기준) · ONNX Runtime GPU · PyTorch GPU · **TensorRT FP32 · TensorRT FP16**.
2. 평가 영상 4편의 사건 평가를 **CPU와 TensorRT FP16으로 각각** 돌려, 속도를 올려도 사건 결과(놓침·잘못 울림·중복)가 그대로인지 봅니다.

**하는 법**
1. 메뉴 **런타임 → 런타임 유형 변경 → `T4 GPU`** → 저장
2. 메뉴 **런타임 → 모두 실행**
3. 끝나면 `gpu_results.zip`이 내려받아집니다. 그 파일을 Claude에게 올려 주세요.

걸리는 시간: 약 20~30분(설치, TensorRT 엔진 만들기, 사건 평가가 대부분). 무료 Colab은 GPU 사용 시간이 제한될 수 있습니다. 중간에 끊기면 다시 **모두 실행**하면 됩니다.

## 1. GPU 확인

In [ ]:
import subprocess
r = subprocess.run(["nvidia-smi"], capture_output=True, text=True)
print(r.stdout or r.stderr)
if r.returncode != 0:
    raise SystemExit("GPU가 없습니다. 메뉴 '런타임 → 런타임 유형 변경'에서 'T4 GPU'를 고르고 저장한 뒤 다시 '모두 실행'하세요.")

## 2. 코드 받기

In [ ]:
%cd /content
!rm -rf sceneproof && git clone -q --depth 1 https://github.com/sokldjs554/sceneproof.git
%cd /content/sceneproof
!git log -1 --format='코드 버전: %h %s'

## 3. 설치 (2~5분)
Colab에 이미 있는 PyTorch는 그대로 쓰고, ONNX Runtime GPU판과 TensorRT를 추가로 설치합니다. 빨간 의존성 경고가 나와도 다음 칸이 실행되면 괜찮습니다.

In [ ]:
!pip -q install -e . --no-deps
!pip -q install "pydantic-settings>=2.4" "grpcio>=1.66" "protobuf>=5.28" psutil onnx
!pip -q uninstall -y onnxruntime
!pip -q install onnxruntime-gpu tensorrt-cu12
!python -c "import onnxruntime as o, tensorrt as t, torch; print('onnxruntime', o.__version__, o.get_available_providers()); print('tensorrt', t.__version__); print('torch', torch.__version__, 'cuda', torch.version.cuda, torch.cuda.get_device_name(0))"

## 4. 모델과 영상 받기

In [ ]:
!ONNX_FROM=official VARIANTS="yolox_nano yolox_tiny yolox_s" bash scripts/fetch_models.sh
!bash scripts/fetch_data.sh
!mkdir -p results

## 5. 속도와 결과 차이 (약 5~10분)
처음 한 번은 TensorRT가 모델마다 엔진을 만드느라 수십 초씩 걸립니다(만드는 시간도 기록됩니다).

In [ ]:
!python scripts/gpu_bench.py --frames 300 --repeats 3 --out results/gpu_bench.json

## 6. 사건 평가: CPU vs TensorRT FP16 (약 10분)
같은 평가 영상 4편을 두 방식으로 분석합니다. 사건 결과가 같으면 'GPU로 빠르게 해도 판정은 그대로'라는 뜻입니다.

In [ ]:
!PYTHONPATH=src python -m sceneproof.eval.run_eval --variant yolox_tiny --out results/event_eval_yolox_tiny_cpu.json > results/event_eval_cpu.log 2>&1; tail -3 results/event_eval_cpu.log
!PYTHONPATH=src python -m sceneproof.eval.run_eval --variant yolox_tiny --backend tensorrt --precision fp16 --out results/event_eval_yolox_tiny_trt_fp16.json > results/event_eval_trt.log 2>&1; tail -3 results/event_eval_trt.log

In [ ]:
import json, os
def load(p):
    try:
        return json.load(open(p))
    except Exception as e:
        print("읽지 못함:", p, e); return None
cpu, trt = load("results/event_eval_yolox_tiny_cpu.json"), load("results/event_eval_yolox_tiny_trt_fp16.json")
keys = ["tp", "fp", "missed", "duplicates", "boundary_hits", "precision", "recall", "f1"]
if cpu and trt:
    print(f"{'':14s}" + "".join(f"{k:>12s}" for k in keys))
    for name, d in (("CPU", cpu), ("TensorRT FP16", trt)):
        print(f"{name:14s}" + "".join(f"{d['aggregate'][k]:>12}" for k in keys))
    for cam in cpu["per_video"]:
        a, b = cpu["per_video"][cam], trt["per_video"].get(cam, {})
        same = all(a.get(k) == b.get(k) for k in ("tp", "fp", "missed", "duplicates"))
        print(cam, "같음" if same else "다름", {k: (a.get(k), b.get(k)) for k in ("tp", "fp", "missed", "duplicates", "start_mae_s", "end_mae_s")})

## 7. 결과 내려받기
`gpu_results.zip`을 Claude에게 올려 주세요. 영상은 들어 있지 않고 숫자(JSON)와 기록만 들어 있습니다.

In [ ]:
!cd results && zip -q -r ../gpu_results.zip . && cd .. && ls -la gpu_results.zip
try:
    from google.colab import files
    files.download("gpu_results.zip")
except Exception as e:
    print("자동 다운로드가 안 되면 왼쪽 파일 창에서 sceneproof/gpu_results.zip을 내려받으세요.", e)